# Scaling up to CNNs
With the basics out of the way, we can start building larger networks. In this notebook, we will introduce to you how to use convolutional and pooling layers.

In [1]:
import torch
import torch.nn as nn
import tonic
from torch.utils.data import DataLoader
from tqdm import tqdm
from spikinggamma.neuron import SpikingGammaNeuron
from spikinggamma.spike.sigma_delta import SigmaDeltaSpikeFunction
from spikinggamma.spike.identity import IdentitySpikeFunction
from spikinggamma.dendrite.split import SplitDendriticFunction
from spikinggamma.dendrite.synapse.linear import LinearPerNeuron
from spikinggamma.signal.default import DefaultSignalFunction
from spikinggamma.spike.base import SpikeConfig
from spikinggamma.dendrite.synapse.conv2d import Conv2dPerNeuron
from spikinggamma.spike.kernel.gamma import GammaKernelFunction, compute_linspace_bucket_decay_rates
from spikinggamma.utils.datasets.dvsgesture import get_dvsgesture_dataset
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


## Loading the data
We use DVS Gesture for this experiment, with 500 timesteps. Caching is used through tonic's `DiskCachedDataset`. This makes the first epoch much slower.

In [2]:
project_dir = Path.cwd().parent
data_path = project_dir / "data"
cache_path = project_dir / "cache"
train_dataset, test_dataset, input_shape, n_outputs, T = get_dvsgesture_dataset(data_path=data_path, cache_path=cache_path, timestep_size=12000)

## Building the network
Building a CNN is very similar to a fully-connected network: replace `LinearPerNeuron` with `Conv2dPerNeuron` in the dendritic function. As in the getting-started notebook, an `IdentitySpikeFunction` first decomposes the input spikes into kernel components, and each neuron layer receives both kernel components and spikes.

`DefaultSignalFunction` applies `pre_act_fn`, the activation, and then `post_act_fn`. We use `pre_act_fn` for layer normalization and `post_act_fn` for pooling in `conv1` and `conv3`. In `conv3`, an `nn.Sequential` also flattens the pooled signal for the fully-connected layer. Each convolutional neuron keeps its own `spike_fn`, so pooling and flattening act on the continuous signal before spike generation.

The final fully-connected layer omits its spike function and returns the continuous output signal for training and evaluation. Resetting and detaching each neuron also handles its spike function; the input spike function is reset and detached separately.

In [3]:
# Hyperparameters
n_kernels = 10
decay_factor = 0.1
spike_config = SpikeConfig(
    base_spike_threshold=0.3,
    use_dynamic_threshold=True
)
bucket_decay_rates = compute_linspace_bucket_decay_rates(n_kernels, decay_factor)

class SpikingGammaNet(nn.Module):
    def __init__(self, 
                 input_shape, 
                 n_outputs,
                 n_kernels,
                 bucket_decay_rates,
                 spike_config):
        super().__init__()
        output_shape_layer1 = (64, 30, 30)
        output_shape_layer2 = (128, 13, 13)
        output_shape_layer3 = (128, 11, 11)
        input_shape_layer4 = (128*5*5,)
        layer4_shape_out = (n_outputs,)

        self.input_fn = IdentitySpikeFunction(GammaKernelFunction(bucket_decay_rates), spike_config)
        self.conv1 = SpikingGammaNeuron(
            dendritic_fn=SplitDendriticFunction(
                output_shape_layer1,
                n_kernels=n_kernels,
                synaptic_function=Conv2dPerNeuron(input_shape[0], 64, kernel_size=7, stride=1, padding=2)
            ),
            signal_fn=DefaultSignalFunction(
                pre_act_fn=nn.LayerNorm(output_shape_layer1),
                post_act_fn=nn.MaxPool2d(kernel_size=2, stride=2),
            ),
            spike_fn=SigmaDeltaSpikeFunction(GammaKernelFunction(bucket_decay_rates), spike_config),
        )

        self.conv2 = SpikingGammaNeuron(
            dendritic_fn=SplitDendriticFunction(
                output_shape_layer2,
                n_kernels=n_kernels,
                synaptic_function=Conv2dPerNeuron(64, 128, kernel_size=7, stride=1, padding=2)
            ),
            signal_fn=DefaultSignalFunction(pre_act_fn=nn.LayerNorm(output_shape_layer2)),
            spike_fn=SigmaDeltaSpikeFunction(GammaKernelFunction(bucket_decay_rates), spike_config),
        )

        self.conv3 = SpikingGammaNeuron(
            dendritic_fn=SplitDendriticFunction(
                output_shape_layer3,
                n_kernels=n_kernels,
                synaptic_function=Conv2dPerNeuron(128, 128, kernel_size=7, stride=1, padding=2)
            ),
            signal_fn=DefaultSignalFunction(
                pre_act_fn=nn.LayerNorm(output_shape_layer3),
                post_act_fn=nn.Sequential(
                    nn.MaxPool2d(kernel_size=2, stride=2),
                    nn.Flatten(start_dim=1),
                ),
            ),
            spike_fn=SigmaDeltaSpikeFunction(GammaKernelFunction(bucket_decay_rates), spike_config),
        )

        self.fc = SpikingGammaNeuron(
            dendritic_fn=SplitDendriticFunction(
                layer4_shape_out,
                n_kernels=n_kernels,
                synaptic_function=LinearPerNeuron(input_shape_layer4[0], layer4_shape_out[0])
            ),
            signal_fn=DefaultSignalFunction()
        )

    def reset(self):
        self.input_fn.reset()
        self.conv1.reset()
        self.conv2.reset()
        self.conv3.reset()
        self.fc.reset()

    def detach(self):
        self.input_fn.detach()
        self.conv1.detach()
        self.conv2.detach()
        self.conv3.detach()
        self.fc.detach()

    def forward(self, s_in):
        y_hat_k_in, s_in = self.input_fn(s_in)
        y_hat_k_conv1, s_conv1 = self.conv1(y_hat_k_in)
        y_hat_k_conv2, s_conv2 = self.conv2(y_hat_k_conv1)
        y_hat_k_conv3, s_conv3 = self.conv3(y_hat_k_conv2)
        y_fc = self.fc(y_hat_k_conv3)
        return y_fc

## Training the network

In [4]:
train_batch_size = 32
test_batch_size = 128
train_dataloader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True, collate_fn=tonic.collation.PadTensors(batch_first=False))
test_dataloader = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=False, collate_fn=tonic.collation.PadTensors(batch_first=False))
model = SpikingGammaNet(input_shape, n_outputs, n_kernels, bucket_decay_rates, spike_config).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

In [5]:
model.train()
n_epochs = 1

for _ in range(n_epochs):
    batch_iter = iter(train_dataloader)
    pbar = tqdm(batch_iter, total=len(train_dataloader))
    for data, labels in pbar:
        data = data.to(device)
        labels = labels.to(device)

        model.reset()
        optimizer.zero_grad()
        total_loss = 0
        for t in range(data.shape[0]):
            y = model(data[t])
            loss = loss_fn(y, labels) * 1/data.shape[0]
            loss.backward()
            total_loss += loss.item()

        optimizer.step()
        pbar.set_description(f"Loss: {total_loss:.4f}")

Loss: 1.8897: 100%|██████████| 34/34 [11:46<00:00, 20.77s/it]


Remember that the first epoch may be slower because of tonic's `DiskCachedDataset` that is enabled by default.

## Testing
As in the getting-started notebook, sum the continuous output signal over time and take the index with the maximum to compute the predictions.

In [6]:
batch_iter = iter(test_dataloader)
model.eval()
correct = 0
total = 0
with torch.no_grad():
    pbar = tqdm(batch_iter, total=len(test_dataloader))
    for data, labels in pbar:
        data = data.to(device)
        labels = labels.to(device)

        model.reset()
        ys = []
        for t in range(data.shape[0]):
            y = model(data[t])
            ys.append(y)
        ys = torch.stack(ys, dim=0)
        predictions = ys.sum(dim=0).argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
accuracy = correct / total
print(f'Test Accuracy: {accuracy * 100:.2f}%')

100%|██████████| 3/3 [00:56<00:00, 18.89s/it]

Test Accuracy: 56.06%
